In [1]:
!ls -la /kaggle/input

total 8
drwxr-xr-x 4 root   root    4096 Feb  5 11:15 .
drwxr-xr-x 5 root   root    4096 Feb  5 11:15 ..
drwxr-xr-x 4 nobody nogroup    0 Feb  5 11:15 sddm-code
drwxr-xr-x 6 nobody nogroup    0 Dec  2  2020 skin-cancer-mnist-ham10000


In [4]:
!ls -la /kaggle/input/sddm-code
!ls -la /kaggle/input/skin-cancer-mnist-ham10000

total 8
drwxr-xr-x 4 nobody nogroup    0 Feb  5 11:15 .
drwxr-xr-x 4 root   root    4096 Feb  5 11:15 ..
drwxr-xr-x 2 nobody nogroup    0 Feb  5 11:15 configs
-rw-r--r-- 1 nobody nogroup  119 Feb  5 11:15 requirements.txt
drwxr-xr-x 2 nobody nogroup    0 Feb  5 11:15 src
total 130148
drwxr-xr-x 6 nobody nogroup        0 Dec  2  2020 .
drwxr-xr-x 4 root   root        4096 Feb  5 11:15 ..
drwxr-xr-x 2 nobody nogroup        0 Dec  2  2020 ham10000_images_part_1
drwxr-xr-x 2 nobody nogroup        0 Dec  2  2020 HAM10000_images_part_1
drwxr-xr-x 2 nobody nogroup        0 Dec  2  2020 ham10000_images_part_2
drwxr-xr-x 2 nobody nogroup        0 Dec  2  2020 HAM10000_images_part_2
-rw-r--r-- 1 nobody nogroup   563277 Dec  2  2020 HAM10000_metadata.csv
-rw-r--r-- 1 nobody nogroup 30807979 Dec  2  2020 hmnist_28_28_L.csv
-rw-r--r-- 1 nobody nogroup 91820383 Dec  2  2020 hmnist_28_28_RGB.csv
-rw-r--r-- 1 nobody nogroup  2537778 Dec  2  2020 hmnist_8_8_L.csv
-rw-r--r-- 1 nobody nogroup  7524968 De

In [5]:
!cp -r /kaggle/input/sddm-code/* /kaggle/working/
%cd /kaggle/working
!ls -la

/kaggle/working
total 24
drwxr-xr-x 5 root root 4096 Feb  5 11:19 .
drwxr-xr-x 5 root root 4096 Feb  5 11:15 ..
drwxr-xr-x 2 root root 4096 Feb  5 11:19 configs
-rw-r--r-- 1 root root  119 Feb  5 11:19 requirements.txt
drwxr-xr-x 2 root root 4096 Feb  5 11:19 src
drwxr-xr-x 2 root root 4096 Feb  5 11:16 .virtual_documents


In [6]:
!pip -q install -r requirements.txt

In [7]:
import torch
print("CUDA:", torch.cuda.is_available())
!nvidia-smi

CUDA: True
Thu Feb  5 11:20:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------------

In [8]:
from pathlib import Path

p = Path("src/data.py")
txt = p.read_text(encoding="utf-8")

# 1) DataConfig: archive_zip -> Optional[str]
txt = txt.replace("archive_zip: str", "archive_zip: Optional[str]")

# 2) signature maybe_extract_archive
txt = txt.replace(
    "def maybe_extract_archive(archive_zip: str, extracted_dir: str) -> None:",
    "def maybe_extract_archive(archive_zip: Optional[str], extracted_dir: str) -> None:"
)

# 3) skip extract if None
txt = txt.replace(
    "    if not os.path.isfile(archive_zip):",
    "    if not archive_zip:\n        return\n\n    if not os.path.isfile(archive_zip):"
)

p.write_text(txt, encoding="utf-8")
print("Patched:", p)


Patched: src/data.py


In [9]:
import yaml, os

SRC_CFG = "configs/ddpm_ham_64.yaml"   # promijeni ako je drugi naziv
OUT_CFG = "configs/ddpm_ham_64_kaggle.yaml"

with open(SRC_CFG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["root"] = "/kaggle/working/data"
cfg["data"]["extracted_dir"] = "/kaggle/input/skin-cancer-mnist-ham10000"
cfg["data"]["archive_zip"] = None

cfg["data"]["image_dir_glob"] = [
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2",
]
cfg["data"]["metadata_glob"] = [
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"
]

cfg["output"]["out_dir"] = "/kaggle/working/runs/ddpm_ham_64_fast"
cfg["output"]["save_images_dir"] = "/kaggle/working/runs/ddpm_ham_64_fast/samples"

# sigurnije: češći checkpoint + sample
cfg["train"]["save_every_steps"] = 2000
cfg["eval"]["val_every_steps"] = 2000
cfg["eval"]["sample_every_steps"] = 2000

os.makedirs("configs", exist_ok=True)
with open(OUT_CFG, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("Wrote:", OUT_CFG)

Wrote: configs/ddpm_ham_64_kaggle.yaml


In [10]:
from src.data import DataConfig, build_datasets
import yaml

cfg = yaml.safe_load(open("configs/ddpm_ham_64_kaggle.yaml", "r", encoding="utf-8"))
train_ds, val_ds, stats = build_datasets(DataConfig(**cfg["data"]), seed=int(cfg.get("seed", 42)))

print(stats)
print("train:", len(train_ds), "val:", len(val_ds))

{'metadata_rows_total': 10015, 'metadata_path': '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv', 'rows_after_label_map': 10015, 'image_dirs_found': ['/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1', '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2'], 'image_dirs_used_for_index': ['/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1', '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2'], 'images_indexed': 10015, 'rows_with_existing_files': 10015, 'train_size': 8989, 'val_size': 1026}
train: 8989 val: 1026


In [11]:
#!python -m src.train --config configs/ddpm_ham_64_kaggle.yaml

2026-02-05 11:22:45.695645: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770290565.872983     197 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770290565.921149     197 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770290566.331100     197 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770290566.331157     197 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770290566.331161     197 computation_placer.cc:177] computation placer alr

In [14]:
#!ls -lah /kaggle/working/runs/ddpm_ham_64_fast/ckpt_0014000.pt

-rw-r--r-- 1 root root 227M Feb  5 14:55 /kaggle/working/runs/ddpm_ham_64_fast/ckpt_0014000.pt


In [16]:
from pathlib import Path
import re

p = Path("/kaggle/working/src/train.py")
txt = p.read_text(encoding="utf-8")

pattern = r'best_val\s*=\s*float\(ckpt\.get\("val_loss",\s*best_val\)\)'
replacement = (
    'ckpt_val = ckpt.get("val_loss", None)\n'
    '        if ckpt_val is not None:\n'
    '            try:\n'
    '                best_val = float(ckpt_val)\n'
    '            except Exception:\n'
    '                pass'
)

new_txt, n = re.subn(pattern, replacement, txt)
print("replacements:", n)
p.write_text(new_txt, encoding="utf-8")

replacements: 1


12679

In [18]:
from pathlib import Path

p = Path("/kaggle/working/src/train.py")
txt = p.read_text(encoding="utf-8")

needle = '        ema.load_state_dict(ckpt["ema"])'
if needle in txt and "ema.to(accelerator.device)" not in txt:
    txt = txt.replace(
        needle,
        needle + '\n        try:\n            ema.to(accelerator.device)\n        except Exception:\n            for i in range(len(ema.shadow_params)):\n                ema.shadow_params[i] = ema.shadow_params[i].to(accelerator.device)\n'
    )

p.write_text(txt, encoding="utf-8")
print("Patched EMA device move")

Patched EMA device move


In [19]:
#!python -m src.train --config configs/ddpm_ham_64_kaggle.yaml --resume /kaggle/working/runs/ddpm_ham_64_fast/ckpt_0014000.pt

2026-02-05 15:16:30.720324: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770304590.741280    1443 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770304590.747807    1443 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770304590.765371    1443 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770304590.765398    1443 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770304590.765401    1443 computation_placer.cc:177] computation placer alr

In [21]:
!cp -r /kaggle/input/sddm-code/src /kaggle/working/src
!mkdir -p /kaggle/working/configs
!cp -r /kaggle/input/sddm-code/configs/* /kaggle/working/configs/
!cp /kaggle/input/sddm-code/requirements.txt /kaggle/working/requirements.txt

!ls -la /kaggle/working/src | head
!ls -la /kaggle/working/configs | head

total 64
drwxr-xr-x 4 root root  4096 Feb  5 17:06 .
drwxr-xr-x 7 root root  4096 Feb  5 15:10 ..
-rw-r--r-- 1 root root 10292 Feb  5 11:21 data.py
-rw-r--r-- 1 root root     0 Feb  5 11:19 __init__.py
-rw-r--r-- 1 root root     0 Feb  5 11:19 __main__.py
-rw-r--r-- 1 root root  1226 Feb  5 11:19 preprocess.py
drwxr-xr-x 2 root root  4096 Feb  5 15:16 __pycache__
-rw-r--r-- 1 root root  4971 Feb  5 11:19 sample.py
drwxr-xr-x 2 root root  4096 Feb  5 17:06 src
total 16
drwxr-xr-x 2 root root 4096 Feb  5 11:21 .
drwxr-xr-x 7 root root 4096 Feb  5 15:10 ..
-rw-r--r-- 1 root root 1227 Feb  5 11:21 ddpm_ham_64_kaggle.yaml
-rw-r--r-- 1 root root 1496 Feb  5 17:07 ddpm_ham_64.yaml


In [ ]:
#import os, yaml, torch
#from PIL import Image
#from tqdm.auto import tqdm
#from diffusers import UNet2DModel, DDIMScheduler

config_path = "/kaggle/working/configs/ddpm_ham_64_kaggle.yaml"   # prilagodi ako ti je drugačije ime
ckpt_path   = "/kaggle/working/runs/ddpm_ham_64_fast/ckpt_best.pt"

cfg = yaml.safe_load(open(config_path, "r", encoding="utf-8"))
image_size = int(cfg["data"]["image_size"])
num_classes = int(cfg["data"]["num_classes"])  # treba biti 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

m = cfg["model"]
base = int(m["base_channels"])
block_out = tuple(base * int(x) for x in m["channel_mults"])
layers = int(m["layers_per_block"])
dropout = float(m.get("dropout", 0.0))
add_attn = bool(m.get("add_attention", True))

n_blocks = len(block_out)
if add_attn:
    mid = n_blocks // 2
    down_types = ["DownBlock2D"] * n_blocks
    up_types = ["UpBlock2D"] * n_blocks
    down_types[mid] = "AttnDownBlock2D"
    up_types[mid] = "AttnUpBlock2D"
else:
    down_types = ["DownBlock2D"] * n_blocks
    up_types = ["UpBlock2D"] * n_blocks

unet = UNet2DModel(
    sample_size=image_size,
    in_channels=3,
    out_channels=3,
    layers_per_block=layers,
    block_out_channels=block_out,
    down_block_types=down_types,
    up_block_types=up_types,
    dropout=dropout,
    class_embed_type="timestep",
    num_class_embeds=num_classes,
).to(device).eval()

state = torch.load(ckpt_path, map_location="cpu")
unet.load_state_dict(state["unet"], strict=True)

# DDIM scheduler
steps = 200  
ddim = DDIMScheduler(
    num_train_timesteps=int(cfg["diffusion"]["num_train_timesteps"]),
    beta_schedule=str(cfg["diffusion"]["beta_schedule"]),
    prediction_type=str(cfg["diffusion"]["prediction_type"]),
    clip_sample=True,   # stabilnije
)
ddim.set_timesteps(steps, device=device)

out_dir = "/kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class"
os.makedirs(out_dir, exist_ok=True)

@torch.no_grad()
def sample_one(cls: int, seed: int):
    g = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn((1, 3, image_size, image_size), generator=g, device=device)
    labels = torch.tensor([cls], dtype=torch.long, device=device)

    for t in ddim.timesteps:
        noise_pred = unet(x, t, class_labels=labels).sample
        x = ddim.step(noise_pred, t, x).prev_sample

    img = ((x.clamp(-1, 1) + 1) / 2).clamp(0, 1)[0]  # C,H,W in [0,1]
    img_u8 = (img * 255.0).round().to(torch.uint8).permute(1, 2, 0).cpu().numpy()
    return Image.fromarray(img_u8)

base_seed = int(cfg.get("seed", 42))
for cls in [0, 1]:
    cls_dir = os.path.join(out_dir, f"class_{cls}")
    os.makedirs(cls_dir, exist_ok=True)
    for i in tqdm(range(100), desc=f"class {cls}"):
        im = sample_one(cls, seed=base_seed + cls*10_000 + i)
        im.save(os.path.join(cls_dir, f"{cls}_{i:03d}.png"))

print("Saved to:", out_dir)

In [23]:
#!zip -r /kaggle/working/generated_100_per_class.zip /kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class
#!ls -lh /kaggle/working/generated_100_per_class.zip

  adding: kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class/ (stored 0%)
  adding: kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class/class_0/ (stored 0%)
  adding: kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class/class_0/0_079.png (deflated 1%)
  adding: kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class/class_0/0_068.png (stored 0%)
  adding: kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class/class_0/0_054.png (stored 0%)
  adding: kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class/class_0/0_050.png (deflated 0%)
  adding: kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class/class_0/0_078.png (stored 0%)
  adding: kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class/class_0/0_046.png (stored 0%)
  adding: kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class/class_0/0_032.png (stored 0%)
  adding: kaggle/working/runs/ddpm_ham_64_fast/generated_100_per_class/class_0/0_076.png (stored 0%)
  adding: ka

In [24]:
!zip -r /kaggle/working/ddpm_ham_64_fast_artifacts.zip \
  /kaggle/working/runs/ddpm_ham_64_fast \
  /kaggle/working/configs \
  /kaggle/working/src \
  /kaggle/working/requirements.txt
!ls -lh /kaggle/working/ddpm_ham_64_fast_artifacts.zip

updating: kaggle/working/runs/ddpm_ham_64_fast/ (stored 0%)
updating: kaggle/working/runs/ddpm_ham_64_fast/ckpt_0002000.pt (deflated 8%)
updating: kaggle/working/runs/ddpm_ham_64_fast/ckpt_0008000.pt (deflated 8%)
updating: kaggle/working/runs/ddpm_ham_64_fast/ckpt_last.pt (deflated 8%)
updating: kaggle/working/runs/ddpm_ham_64_fast/ckpt_0010000.pt (deflated 8%)
updating: kaggle/working/runs/ddpm_ham_64_fast/ckpt_0012000.pt (deflated 8%)
updating: kaggle/working/runs/ddpm_ham_64_fast/ckpt_0004000.pt (deflated 8%)
updating: kaggle/working/runs/ddpm_ham_64_fast/ckpt_best.pt (deflated 8%)
updating: kaggle/working/runs/ddpm_ham_64_fast/ckpt_0006000.pt (deflated 8%)
updating: kaggle/working/runs/ddpm_ham_64_fast/ckpt_0014000.pt (deflated 8%)
updating: kaggle/working/runs/ddpm_ham_64_fast/samples/ (stored 0%)
updating: kaggle/working/runs/ddpm_ham_64_fast/samples/sample_step_0002000.png (deflated 0%)
updating: kaggle/working/runs/ddpm_ham_64_fast/samples/sample_step_0010000.png (deflated 0%)